In [ ]:
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
from nltk.corpus import stopwords

print(stopwords.words('english')[:10])

In [ ]:
import pandas as pd
import re
import contractions

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))

# Keep negations because they are important
stop_words = stop_words - {
    'not', 'no', 'nor',
    "don't", "didn't", "won't",
    "isn't", "aren't"
}

lemmatizer = WordNetLemmatizer()

# Chat abbreviations
chat_words = {
    'asap': 'as soon as possible',
    'pls': 'please',
    'plz': 'please',
    'u': 'you',
    'ur': 'your',
    'btw': 'by the way',
    'idk': 'i do not know',
    'imo': 'in my opinion',
    'thx': 'thanks',
    'msg': 'message',

    'acct': 'account',
    'cc': 'credit card',
    'amt': 'amount',
    'txn': 'transaction',
    'bk': 'bank',
    'svc': 'service',
    'cust': 'customer'
}
#
def preprocess_text(text):

    # Handle missing values
    if pd.isna(text):
        return ""

    # Lowercase
    text = text.lower()

    # Expand contractions
    text = contractions.fix(text)

    # Replace chat abbreviations
    words = text.split()

    words = [
        chat_words[word]
        if word in chat_words
        else word
        for word in words
    ]

    text = " ".join(words)

    #Replace emails, URLs, and phone numbers with placeholders
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)
    
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    
    text = re.sub(r'\b\d{10,}\b', ' PHONE ', text)

    # Remove punctuation
    text = re.sub(
        r'[^\w\s]',
        '',
        text
    )

    # Remove extra spaces
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    # Remove stopwords
    words = text.split()

    words = [
        word
        for word in words
        if word not in stop_words
    ]

    # Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)

In [ ]:

df_u = pd.read_csv(r'D:\Data_Science\projects\CFPB_Complaint_Intelligence_pipeline\notebooks\urgent\df_u.csv')
df_u.columns

In [ ]:
# Check target column
print(df_u['Timely response?'].isnull().sum())

df_u['Timely response?'] = (
    df_u['Timely response?']
    .str.strip()
    .str.capitalize()
)


In [ ]:
df_u['narrative'].sample(10, random_state=42)

In [ ]:
df_u['clean_narrative'] = df_u['narrative'].apply(preprocess_text)

In [ ]:
df_u[['narrative', 'clean_narrative']].head()

In [ ]:
# Create a new dataframe with only the required columns
df_u_clean = df_u[['clean_narrative', 'Timely response?']]

# Save it as a CSV file
df_u_clean.to_csv(r"D:\Data_Science\projects\CFPB_Complaint_Intelligence_pipeline\notebooks\urgent\df_u_clean.csv",index=False)

print("CSV file saved successfully!")

In [ ]:
import pandas as pd

# Load the CSV
df_u_clean = pd.read_csv("df_u_clean.csv")

# Rename the column
df_u_clean.rename(
    columns={'clean_narrative': 'narrative'},
    inplace=True
)

# Save it again (overwrite the old file)
df_u_clean.to_csv(
    "df_u_clean.csv",
    index=False
)

print("Column renamed successfully!")

In [ ]:
df_u.shape()